In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from functools import reduce

In [0]:
# Definir rango de periodos
codmes_data_ini = 202307
codmes_data_fin = 202512

# 1. Universo

In [0]:
df_universo = spark.table("catalog_cemm_expl_005_bcp_prod.bcp_expl_mmgr_fabr.t74985_base_flg_cuenta").filter(
    (F.col("codclavepartycli").isNotNull()) &
    (F.col("codmes").between(codmes_data_ini, codmes_data_fin))
)

df_universo.limit(10).display()

In [0]:
df_universo.groupBy("codmes").agg(
    F.count("codclavecta").alias("num_cuentas"),
    F.countDistinct("codclavecta").alias("num_cuentas_unicas"),
    F.countDistinct("codclavepartycli").alias("num_clientes_unicos"),
).orderBy("codmes").display()

In [0]:
df_universo = df_universo.groupBy("codmes", "codclavepartycli").agg(
    F.count("codclavecta").alias("num_cuentas_unicas"),
    F.max("target_em31_bloq_3m").alias("target_em31_bloq_3m"),
    F.first("destipnivelvulnerabilidadcli").alias("destipnivelvulnerabilidadcli")
).withColumn(
    "flg_vul",
    F.lit(1)
)

df_universo.display()

# 2. Agregamos la PD bhv

In [0]:
df_bhv = spark.table("catalog_cemm_expl_005_bcp_prod.bcp_expl_mmgr_fabr.T36436x_bhv_port_d_mod_dev_gb_out_v3_history_vu").filter((F.col("codmes") >= F.lit(codmes_data_ini)) & (F.col("codmes") <= F.lit(codmes_data_fin)))

df_bhv = df_bhv.withColumn("XB_30_12", F.log(F.col("numpd") / (1 - F.col("numpd"))))

intercept = -1.4383977166087802
coef_XB_30_12 = 1.1506739569389508

df_bhv = df_bhv.withColumn("XB_30_3", intercept + coef_XB_30_12 * F.col("XB_30_12"))
df_bhv = df_bhv.withColumn("PD_30_3", F.exp(F.col("XB_30_3"))/(1+F.exp(F.col("XB_30_3"))))
df_bhv = df_bhv.withColumn("SC_30_3", F.when(F.col("XB_30_3").isNotNull(), F.greatest(F.lit(3), F.round(174.25 - 57.708 * F.col("XB_30_3"), 1))))

In [0]:
df_bhv.display()

In [0]:
df_bhv = df_bhv.withColumn(
    "destipovulnerabilidad",
    F.when(F.col("PD_30_3") < 0.02, F.lit("03. No Vulnerable"))
    .when((F.col("PD_30_3") >= 0.02) & (F.col("PD_30_3") < 0.07), F.lit("02. En observación"))
    .otherwise(F.lit("01. Vulnerable"))
)

In [0]:
df_bhv = df_bhv.withColumn("flg_bhv", F.lit(1))

In [0]:
df_bhv.groupBy("codmes").agg(
    F.count("codclavepartycli").alias("num_clientes"),
    F.countDistinct("codclavepartycli").alias("num_clientes_unicos")
).orderBy("codmes").display()

# 3. Cruce de tablas

In [0]:
df_final = df_universo.select("codmes", "codclavepartycli", "flg_vul").join(
    df_bhv.select("codmes", "codclavepartycli", "numpd", "XB_30_3", "PD_30_3", "SC_30_3", "destipovulnerabilidad", "flg_bhv"),
    on=["codmes", "codclavepartycli"],
    how="full_outer"
)

df_final.limit(10).display()

In [0]:
df_final_2 = df_universo.select("codmes", "codclavepartycli").join(
    df_bhv.select("codmes", "codclavepartycli", "numpd", "XB_30_3", "PD_30_3", "SC_30_3", "destipovulnerabilidad"),
    on=["codmes", "codclavepartycli"],
    how="inner"
)

df_final_2.display() #==============> ESTA TABLA SE COMPARTE A IMPLEMENTACIÓN

# 4. Tabla Final

In [0]:
# VERSION ACTUAL PARA IMPLEMENTACION (pasarle con inner)
spark.sql("USE CATALOG catalog_cemm_expl_005_bcp_prod")

df_final_2.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bcp_expl_mmgr_fabr.t74985_punt_vul_bhv_implementacion_2")